## Predict Temprature/Weather Condition

In [3]:
import csv
from datetime import datetime

In [4]:
def load_observations():
    """
    Read all rows from the CSV file and return them as a list of dicts.
    """
    observations = []
    with open('observations.csv', mode="r", newline="\n") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Cast numeric columns so math works later
            row["temperature_c"]   = float(row["temperature_c"])
            row["humidity_pct"]    = float(row["humidity_pct"])
            row["wind_speed_kmh"]  = float(row["wind_speed_kmh"])
            observations.append(row)
    return observations

In [2]:
def predict_tomorrow(observations):
    if not observations:
        print("No observations recorded yet.")
        return

    current_month = datetime.today().month
    same_month_obs = []
    for obs in observations:
        month = int(obs["date"].split("-")[0])
        if month == current_month:
            same_month_obs.append(obs)

    if not same_month_obs:
        print("Not enough historical data for this month yet.")
        return

    total_temp = 0
    for obs in same_month_obs:
        total_temp = total_temp + obs["temperature_c"]
    avg_temp = round(total_temp / len(same_month_obs), 1)

    condition_counts = {}
    for obs in same_month_obs:
        condition = obs["condition"]
        if condition not in condition_counts:
            condition_counts[condition] = 0
        condition_counts[condition] = condition_counts[condition] + 1

    likely_condition = ""
    highest_count = 0
    for condition, count in condition_counts.items():
        if count > highest_count:
            highest_count = count
            likely_condition = condition

    current_month_name = datetime.today().strftime("%B")

    print("Tomorrow's Prediction (based on historical data)")
    print("Month: " + current_month_name)
    print("Average temperature: " + str(avg_temp) + "C")
    print("Likely condition: " + likely_condition)
    print("This is an estimate based on past patterns, not a real forecast.")

In [5]:
obs = load_observations()
predict_tomorrow(obs)

Tomorrow's Prediction (based on historical data)
Month: April
Average temperature: 22.0C
Likely condition: Sunny
This is an estimate based on past patterns, not a real forecast.


In [7]:
!pip install scikit-learn numpy

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.0 MB 4.8 MB/s eta 0:00:02
   ------------- -------------------------- 2.6/8.0 MB 7.5 MB/s eta 0:00:01
   ---------------------- ----------------- 4.5/8.0 MB 7.9 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 10.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   -- ------------------------------------- 2.6/36.5 MB 11.6 MB/s eta 0:00:03
   ----- ---------------------------------- 5.2/36.5 MB 12.3 MB/s eta 0:00:03
   ------------- -------------------------- 12.1/36.5 MB 18.9 MB/s eta 0:00:02
   ---------------------- ----------------- 20.7/36.5 MB 24.2 MB/s eta 0:00:01
   ------------------------- -------------- 23.6/36.5 MB 25.8 MB/s eta 0:00:01
   -------------------------------- ------- 29.6/36.5 MB 23.2 MB/s eta 0:00:01
   ------------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
import numpy as np

def predict_tomorrow(observations):
    if not observations:
        print("No observations recorded yet.")
        return

    # --- Step 1: Prepare the data ---
    months = []
    temperatures = []
    conditions = []

    for obs in observations:
        month = int(obs["date"].split("-")[0])
        months.append(month)
        temperatures.append(obs["temperature_c"])
        conditions.append(obs["condition"])

    # sklearn needs a 2D array for input, so we reshape
    # [[1], [2], [3]] instead of [1, 2, 3]
    months_2d = [[m] for m in months]

    # --- Step 2: Predict temperature (a number) ---
    temp_model = LinearRegression()
    temp_model.fit(months_2d, temperatures)

    current_month = datetime.today().month
    predicted_temp = temp_model.predict([[current_month]])
    predicted_temp = round(predicted_temp[0], 1)

    # --- Step 3: Predict condition (a category like "Sunny", "Rainy") ---
    condition_model = KNeighborsClassifier(n_neighbors=3)
    condition_model.fit(months_2d, conditions)

    predicted_condition = condition_model.predict([[current_month]])
    predicted_condition = predicted_condition[0]

    # --- Step 4: Print results ---
    current_month_name = datetime.today().strftime("%B")

    print("Tomorrow's Prediction (based on historical data)")
    print("Month: " + current_month_name)
    print("Predicted temperature: " + str(predicted_temp) + "C")
    print("Likely condition: " + str(predicted_condition))
    print("This is an estimate based on a real forecast.")

In [11]:
predict_tomorrow(obs)

Tomorrow's Prediction (based on historical data)
Month: April
Predicted temperature: 19.3C
Likely condition: Sunny
This is an estimate based on a real forecast.
